In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import os
import sys

os.environ["TORCHDYNAMO_INLINE_INBUILT_NN_MODULES"] = "1"
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["MUJOCO_GL"] = "egl"


import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.amp import autocast, GradScaler
from tensordict import TensorDict, from_module

torch.autograd.set_detect_anomaly(True)
torch.set_float32_matmul_precision("high")

from fast_td3.fast_td3_utils import (
    EmpiricalNormalization,
)

from fast_td3.actors import ActorEGNN, ActorEGNN_V2, Actor

In [3]:
from fast_td3.hyperparams import HumanoidBenchArgs

robot = "h1"

args = HumanoidBenchArgs(
    env_name=f"{robot}-balance_simple-v0",
    total_timesteps=50000,
    render_interval=5000,
    eval_interval=1000,
    num_envs=16,
    batch_size=8192,
    actor_hidden_dim=384,
)

In [4]:
amp_enabled = args.amp and args.cuda and torch.cuda.is_available()
amp_device_type = (
    "cuda"
    if args.cuda and torch.cuda.is_available()
    else "mps" if args.cuda and torch.backends.mps.is_available() else "cpu"
)
amp_dtype = torch.bfloat16 if args.amp_dtype == "bf16" else torch.float16

scaler = GradScaler(enabled=amp_enabled and amp_dtype == torch.float16)


random.seed(args.seed)
np.random.seed(args.seed)
torch.manual_seed(args.seed)
torch.backends.cudnn.deterministic = args.torch_deterministic

if not args.cuda:
    device = torch.device("cpu")
else:
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{args.device_rank}")
    elif torch.backends.mps.is_available():
        device = torch.device(f"mps:{args.device_rank}")
    else:
        raise ValueError("No GPU available")
print(f"Using device: {device}")

Using device: cuda:0


In [5]:
from fast_td3.environments.humanoid_bench_env import HumanoidBenchEnv

env_type = "humanoid_bench"
eval_envs = HumanoidBenchEnv(args.env_name, args.num_envs, device=device)
render_env = HumanoidBenchEnv(args.env_name, 1, render_mode="rgb_array", device=device)

n_act = eval_envs.num_actions
n_obs = eval_envs.num_obs if type(eval_envs.num_obs) == int else eval_envs.num_obs[0]
if eval_envs.asymmetric_obs:
    n_critic_obs = (
        eval_envs.num_privileged_obs
        if type(eval_envs.num_privileged_obs) == int
        else eval_envs.num_privileged_obs[0]
    )
else:
    n_critic_obs = n_obs
action_low, action_high = -1.0, 1.0

In [6]:
checkpoint_path = None
checkpoint_path = "./models/egnn_v2_h1-balance_simple-v0_16envs_1000001steps_745fec_840000.pt"
obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
xanchor_normalizer = nn.Identity()
# Actor setup
actor = ActorEGNN_V2(
    num_envs=16,
    batch_size=args.batch_size,
    device=device,
    hidden_dim=64,
    n_layers=4,
    act_fn="relu",
    robot=robot,
    env_name=args.env_name,
	  tanh=True,
	  coords_agg="sum",
)


torch_checkpoint = torch.load(
    f"{checkpoint_path}", map_location=device, weights_only=False
)
obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
pretrained_state_dict = torch_checkpoint["actor_state_dict"]    
actor.load_state_dict(torch_checkpoint["actor_state_dict"])

normalize_obs = obs_normalizer.forward
normalize_xanchor = xanchor_normalizer.forward

In [7]:
# checkpoint_path = "./models/mlp_h1-balance_simple-v0_16envs_1000001steps_aef324_final.pt"
# obs_normalizer = EmpiricalNormalization(shape=n_obs, device=device)
# xanchor_normalizer = nn.Identity()
# normalize_obs = obs_normalizer.forward
# normalize_xanchor = xanchor_normalizer.forward

# actor = Actor(
#     n_obs=n_obs,
#     n_act=n_act,
#     num_envs=args.num_envs,
#     device=device,
#     init_scale=args.init_scale,
#     hidden_dim=args.actor_hidden_dim,
# )

# torch_checkpoint = torch.load(
#     f"{checkpoint_path}", map_location=device, weights_only=False
# )
# obs_normalizer.load_state_dict(torch_checkpoint["obs_normalizer_state"])
# # xanchor_normalizer.load_state_dict(torch_checkpoint["xanchor_normalizer_state"])
# pretrained_state_dict = torch_checkpoint["actor_state_dict"]    
# actor.load_state_dict(torch_checkpoint["actor_state_dict"])

# normalize_obs = obs_normalizer.forward
# normalize_xanchor = xanchor_normalizer.forward

# print("actor parameters:", sum(p.numel() for p in actor.parameters()))

In [8]:
def evaluate():
    obs_normalizer.eval()
    xanchor_normalizer.eval()
    num_eval_envs = eval_envs.num_envs
    episode_returns = torch.zeros(num_eval_envs, device=device)
    episode_lengths = torch.zeros(num_eval_envs, device=device)
    done_masks = torch.zeros(num_eval_envs, dtype=torch.bool, device=device)
    
    obs, xanchor = eval_envs.reset(random_position=False, random_orientation=True)
    for _ in range(eval_envs.max_episode_steps):
        with torch.no_grad(), autocast(
            device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
        ):  
            obs = normalize_obs(obs)
            xanchor = normalize_xanchor(xanchor)
            actions = actor(obs, xanchor)

        next_obs, rewards, dones, _ , next_xanchor = eval_envs.step(actions.float())
        episode_returns = torch.where(
            ~done_masks, episode_returns + rewards, episode_returns
        )
        episode_lengths = torch.where(~done_masks, episode_lengths + 1, episode_lengths)
        done_masks = torch.logical_or(done_masks, dones)
        if done_masks.all():
            break
        obs = next_obs
        xanchor = next_xanchor

    obs_normalizer.train()
    xanchor_normalizer.train()
    return episode_returns.mean().item(), episode_lengths.mean().item()

In [9]:
evaluate()

tensor([[23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427],
        [23.9598,  2.9949, -1.6451,  9.5427]], device='cuda:0')
tensor([[22.9049,  1.3662, -2.8941,  9.7680],
        [23.0656,  1.3201, -3.0301,  9.7670],
        [23.0276,  1.3058, -3.0156,  9.7613],
        [22.8832,  1.3731, -2.8775,  9.7705],
        [22.9641,  1.3456, -2.9395,  9.7677],
        [23.0490

(15.905925750732422, 20.5625)

In [14]:
import tempfile
import imageio
import base64
from IPython.display import display, HTML


def frames_to_video_html(frames, fps=30):
	"""
	Convert a list of numpy arrays to an HTML5 video element.

	Args:
		frames (list): List of numpy arrays representing video frames
		fps (int): Frames per second for the video

	Returns:
		HTML object containing the video element
	"""
	# Create a temporary file to store the video
	with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp_file:
		temp_filename = temp_file.name

	# Save frames as video
	imageio.mimsave(temp_filename, frames, fps=fps)

	# Read the video file and encode it to base64
	with open(temp_filename, "rb") as f:
		video_data = f.read()
	video_b64 = base64.b64encode(video_data).decode("utf-8")

	# Create HTML video element
	video_html = f"""
	<video width="640" height="480" controls>
		<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
		Your browser does not support the video tag.
	</video>
	"""

	# Clean up the temporary file
	os.unlink(temp_filename)

	return HTML(video_html)


def render_with_rollout():
	obs_normalizer.eval()
	xanchor_normalizer.eval()

	# Quick rollout for rendering
	obs, xanchor = render_env.reset(random_position=False, random_orientation=True)
	renders = []

	qpos_max = []
	qvel_max = []
	xanchor_max = []
	qpos_min = []
	qvel_min = []
	xanchor_min = []
	object_vel_max = []
	object_vel_min = []
 
	quat = []
	quat_normalized = []

	for i in range(render_env.max_episode_steps):
		with torch.no_grad(), autocast(
			device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
		):
			xanchor2 = xanchor[:, :] - xanchor[:, 0]
			xanchor_max.append(xanchor2.max().item())
			qpos_max.append(obs[:, 7:26].max().item())
			qvel_max.append(obs[:, 32:].max().item())
			qpos_min.append(obs[:, 7:26].min().item())
			qvel_min.append(obs[:, 32:].min().item())
			xanchor_min.append(xanchor2.min().item())
			object_vel_max.append(obs[:, 26:32].max().item())
			object_vel_min.append(obs[:, 26:32].min().item())
   
			quat.append(obs[:, 3:7])
			obs = normalize_obs(obs)
			quat_normalized.append(obs[:, 3:7])
   
			xanchor = normalize_xanchor(xanchor)
			actions = actor(obs, xanchor)

		next_obs, _, done, _, next_xanchor = render_env.step(actions.float())
		if i % 2 == 0:
			if env_type == "humanoid_bench":
				renders.append(render_env.render())
			else:
				renders.append(render_env.state)
		if done.any():
			break
		obs = next_obs
		xanchor = next_xanchor

	if env_type == "mujoco_playground":
		renders = render_env.render_trajectory(renders)

	obs_normalizer.train()
	xanchor_normalizer.train()
	video_html = frames_to_video_html(renders, fps=30)
	display(video_html)

	# import matplotlib.pyplot as plt
	# plt.plot(qpos_max)
	# plt.plot(qvel_max)
	# plt.plot(xanchor_max)
	# plt.plot(qpos_min)
	# plt.plot(qvel_min)
	# plt.plot(xanchor_min)
	# plt.show()

	# print(max(qpos_max))
	# print(min(qpos_min))
	# print(max(qvel_max))
	# print(min(qvel_min))
	# print(max(xanchor_max))
	# print(min(xanchor_min))
	# print(max(object_vel_max))
	# print(min(object_vel_min))
 
	print(quat[:15])
	print(quat_normalized[:15])
 
render_with_rollout()

tensor([[23.9598,  2.9949, -1.6451,  9.5427]], device='cuda:0')
tensor([[22.8513,  1.3776, -2.8552,  9.7721]], device='cuda:0')
tensor([[20.7713, -0.9210, -4.7805, 10.0961]], device='cuda:0')
tensor([[20.4907, -1.8703, -6.3917,  9.8030]], device='cuda:0')
tensor([[20.2500, -2.4032, -7.6246,  9.4533]], device='cuda:0')
tensor([[19.5933, -3.1186, -8.3862,  9.1290]], device='cuda:0')
tensor([[19.8988, -4.0749, -9.1368,  8.5196]], device='cuda:0')
tensor([[19.0678, -4.5133, -9.4458,  8.1428]], device='cuda:0')
tensor([[17.1852, -4.1806, -9.2362,  7.8568]], device='cuda:0')
tensor([[16.3961, -4.0093, -9.2864,  7.2795]], device='cuda:0')
tensor([[17.1907, -3.6224, -9.4368,  6.3063]], device='cuda:0')
tensor([[17.5759, -2.0173, -9.0579,  5.4597]], device='cuda:0')
tensor([[18.0825,  0.7242, -8.5240,  4.8066]], device='cuda:0')
tensor([[18.7641,  3.2868, -8.7777,  4.5358]], device='cuda:0')
tensor([[ 17.6499,   2.3117, -10.7862,   4.8384]], device='cuda:0')
tensor([[ 16.0531,   0.5627, -13.350

[tensor([[0.7071, 0.0000, 0.0000, 0.7071]], device='cuda:0'), tensor([[ 0.7253,  0.0044, -0.0055,  0.6884]], device='cuda:0'), tensor([[ 0.7588,  0.0144, -0.0137,  0.6510]], device='cuda:0'), tensor([[ 0.7571,  0.0383, -0.0421,  0.6508]], device='cuda:0'), tensor([[ 0.7586,  0.0542, -0.0729,  0.6452]], device='cuda:0'), tensor([[ 0.7624,  0.0507, -0.0814,  0.6400]], device='cuda:0'), tensor([[ 0.7401,  0.0436, -0.0861,  0.6656]], device='cuda:0'), tensor([[ 0.7459,  0.0366, -0.0878,  0.6592]], device='cuda:0'), tensor([[ 0.7777,  0.0263, -0.0852,  0.6223]], device='cuda:0'), tensor([[ 0.7807,  0.0123, -0.0810,  0.6195]], device='cuda:0'), tensor([[ 7.4364e-01, -5.8431e-04, -7.3777e-02,  6.6450e-01]], device='cuda:0'), tensor([[ 0.7220, -0.0088, -0.0647,  0.6889]], device='cuda:0'), tensor([[ 0.7019, -0.0166, -0.0526,  0.7101]], device='cuda:0'), tensor([[ 0.6767, -0.0275, -0.0351,  0.7349]], device='cuda:0'), tensor([[ 0.6868, -0.0383, -0.0166,  0.7257]], device='cuda:0')]
[tensor([[-1

In [11]:
import tempfile
import imageio
import base64
from IPython.display import display, HTML


def frames_to_video_html(frames, fps=30):
	"""
	Convert a list of numpy arrays to an HTML5 video element.

	Args:
		frames (list): List of numpy arrays representing video frames
		fps (int): Frames per second for the video

	Returns:
		HTML object containing the video element
	"""
	# Create a temporary file to store the video
	with tempfile.NamedTemporaryFile(suffix=".mp4", delete=False) as temp_file:
		temp_filename = temp_file.name

	# Save frames as video
	imageio.mimsave(temp_filename, frames, fps=fps)

	# Read the video file and encode it to base64
	with open(temp_filename, "rb") as f:
		video_data = f.read()
	video_b64 = base64.b64encode(video_data).decode("utf-8")

	# Create HTML video element
	video_html = f"""
	<video width="640" height="480" controls>
		<source src="data:video/mp4;base64,{video_b64}" type="video/mp4">
		Your browser does not support the video tag.
	</video>
	"""

	# Clean up the temporary file
	os.unlink(temp_filename)

	return HTML(video_html)


def render_with_rollout():
	obs_normalizer.eval()
	xanchor_normalizer.eval()

	# Quick rollout for rendering
	obs, xanchor = render_env.reset(random_orientation=False)
	renders = [render_env.render()]

	qpos_max = []
	qvel_max = []
	xanchor_max = []
	qpos_min = []
	qvel_min = []
	xanchor_min = []
	object_vel_max = []
	object_vel_min = []
 
	quat = []
	quat_normalized = []

	for i in range(render_env.max_episode_steps):
		with torch.no_grad(), autocast(
			device_type=amp_device_type, dtype=amp_dtype, enabled=amp_enabled
		):
			xanchor2 = xanchor[:, :] - xanchor[:, 0]
			xanchor_max.append(xanchor2.max().item())
			qpos_max.append(obs[:, 7:26].max().item())
			qvel_max.append(obs[:, 32:].max().item())
			qpos_min.append(obs[:, 7:26].min().item())
			qvel_min.append(obs[:, 32:].min().item())
			xanchor_min.append(xanchor2.min().item())
			object_vel_max.append(obs[:, 26:32].max().item())
			object_vel_min.append(obs[:, 26:32].min().item())
   
			quat.append(obs[:, 3:7])
			# obs = normalize_obs(obs)
			quat_normalized.append(obs[:, 3:7])
   
			xanchor = normalize_xanchor(xanchor)
			actions = actor(obs, xanchor)

		next_obs, _, done, _, next_xanchor = render_env.step(actions.float())
		if i % 2 == 0:
			if env_type == "humanoid_bench":
				renders.append(render_env.render())
			else:
				renders.append(render_env.state)
		if done.any():
			break
		obs = next_obs
		xanchor = next_xanchor

	if env_type == "mujoco_playground":
		renders = render_env.render_trajectory(renders)

	obs_normalizer.train()
	xanchor_normalizer.train()
	video_html = frames_to_video_html(renders, fps=30)
	display(video_html)

	# import matplotlib.pyplot as plt
	# plt.plot(qpos_max)
	# plt.plot(qvel_max)
	# plt.plot(xanchor_max)
	# plt.plot(qpos_min)
	# plt.plot(qvel_min)
	# plt.plot(xanchor_min)
	# plt.show()

	# print(max(qpos_max))
	# print(min(qpos_min))
	# print(max(qvel_max))
	# print(min(qvel_min))
	# print(max(xanchor_max))
	# print(min(xanchor_min))
	# print(max(object_vel_max))
	# print(min(object_vel_min))
 
	print(quat[:15])
	print(quat_normalized[:15])
 
render_with_rollout()

tensor([[1., 0., 0., 0.]], device='cuda:0')
tensor([[ 0.9980, -0.0052, -0.0168,  0.0605]], device='cuda:0')
tensor([[ 0.9959, -0.0065, -0.0463,  0.0771]], device='cuda:0')
tensor([[ 0.9983, -0.0109, -0.0517,  0.0260]], device='cuda:0')
tensor([[ 0.9987, -0.0078, -0.0454,  0.0217]], device='cuda:0')
tensor([[ 0.9987, -0.0073, -0.0432, -0.0277]], device='cuda:0')
tensor([[ 9.9895e-01,  9.7818e-04, -1.8758e-02, -4.1734e-02]], device='cuda:0')
tensor([[ 0.9980,  0.0046,  0.0217, -0.0596]], device='cuda:0')
tensor([[ 0.9947,  0.0073,  0.0781, -0.0658]], device='cuda:0')
tensor([[ 0.9806,  0.0069,  0.1789, -0.0796]], device='cuda:0')
tensor([[ 0.9481,  0.0118,  0.3077, -0.0789]], device='cuda:0')
tensor([[ 0.9140,  0.0541,  0.3884, -0.1038]], device='cuda:0')
tensor([[ 0.8919,  0.1046,  0.4247, -0.1149]], device='cuda:0')
tensor([[ 0.8693,  0.1599,  0.4435, -0.1483]], device='cuda:0')
tensor([[ 0.8526,  0.1975,  0.4546, -0.1656]], device='cuda:0')
tensor([[ 0.8392,  0.2161,  0.4652, -0.1805]

[tensor([[1., 0., 0., 0.]], device='cuda:0'), tensor([[ 0.9980,  0.0074, -0.0179, -0.0596]], device='cuda:0'), tensor([[ 0.9966,  0.0137, -0.0313, -0.0751]], device='cuda:0'), tensor([[ 0.9985,  0.0167, -0.0474, -0.0224]], device='cuda:0'), tensor([[ 0.9977,  0.0159, -0.0631, -0.0170]], device='cuda:0'), tensor([[ 0.9972,  0.0130, -0.0648,  0.0336]], device='cuda:0'), tensor([[ 0.9966,  0.0087, -0.0666,  0.0485]], device='cuda:0'), tensor([[ 0.9945,  0.0099, -0.0794,  0.0677]], device='cuda:0'), tensor([[ 0.9918,  0.0115, -0.1016,  0.0769]], device='cuda:0'), tensor([[ 0.9872,  0.0128, -0.1273,  0.0947]], device='cuda:0'), tensor([[ 0.9841,  0.0101, -0.1468,  0.0995]], device='cuda:0'), tensor([[ 0.9755, -0.0145, -0.1725,  0.1361]], device='cuda:0'), tensor([[ 0.9662, -0.0426, -0.1976,  0.1598]], device='cuda:0'), tensor([[ 0.9517, -0.0705, -0.2158,  0.2066]], device='cuda:0'), tensor([[ 0.9405, -0.0865, -0.2320,  0.2327]], device='cuda:0')]
[tensor([[1., 0., 0., 0.]], device='cuda:0')